# TimeSeriesTable

`TimeSeriesTable` stores multiple co-indexed time series as named columns, each with its own metadata — unit, data type, and geographic location. It is fully interoperable with pandas, NumPy, Polars, and PyArrow.

Key characteristics:

- **Named columns**: each value column is an independent signal with its own metadata
- **Shared timestamps**: all columns share a `valid_time` index
- **Per-column metadata**: independent units, data types, and locations per column (length-1 lists broadcast to all columns)
- **Spatial filtering**: select columns by geographic proximity using `GeoLocation`
- **Format interop**: `from_pandas()` / `to_pandas()`, `to_polars()`, `to_numpy()`, `to_pyarrow()`

In [1]:
import pandas as pd
import polars as pl
import numpy as np

from timedatamodel import TimeSeries, TimeSeriesTable, DataType, Frequency, GeoLocation, GeoArea

## Creating from a pandas DataFrame

`TimeSeriesTable.from_pandas()` accepts a DataFrame with a `valid_time` column (or as index) and one or more value columns. Per-column metadata (units, data types, locations) is passed as lists — a length-1 list broadcasts the same value to all columns.

In [2]:
rng = np.random.default_rng(42)
dates = pd.date_range("2024-01-15", periods=24, freq="1h", tz="UTC")

df = pd.DataFrame({
    "valid_time":   dates,
    "wind_speed":   (8.0 + 4.0 * np.sin(np.linspace(0, 2 * np.pi, 24)) + rng.normal(0, 0.5, 24)).round(2),
    "temperature":  (-2.0 + 3.0 * np.sin(np.linspace(0, np.pi, 24)) + rng.normal(0, 0.2, 24)).round(2),
    "solar_power":  np.clip(80 * np.sin(np.linspace(-0.3, np.pi + 0.3, 24)) + rng.normal(0, 2, 24), 0, None).round(2),
})

table = TimeSeriesTable.from_pandas(
    df,
    frequency=Frequency.PT1H,
    units=["m/s", "degC", "MW"],
    data_types=[DataType.OBSERVATION, DataType.OBSERVATION, DataType.OBSERVATION],
    timezone="Europe/Stockholm",
)
table

TimeSeriesTable
┌───────────────────────────────────────────────────────────┐
│  Columns:          wind_speed, temperature, solar_power   │
│  Shape:            24 × 3                                 │
│  Frequency:        PT1H                                   │
│  Timezone:         Europe/Stockholm                       │
│  Unit:             m/s, degC, MW                          │
│  Data type:        OBSERVATION, OBSERVATION, OBSERVATION  │
├───────────────────────────────────────────────────────────┤
│                    wind_speed  temperature  solar_power   │
│  2024-01-15 00:00        8.15        -2.09          0.0   │
│  2024-01-15 01:00        8.56        -1.66          0.0   │
│  2024-01-15 02:00       10.45        -1.08         2.61   │
│  ...                      ...          ...          ...   │
│  2024-01-15 21:00        5.58        -1.15         1.65   │
│  2024-01-15 22:00        7.53        -1.42          0.0   │
│  2024-01-15 23:00        7.92        -1.96          0.0   │
└───────────────────────────────────────────────────────────┘

## Inspecting table properties

In [3]:
print(f"column_names: {table.column_names}")
print(f"n_columns:    {table.n_columns}")
print(f"num_rows:     {table.num_rows}")
print(f"has_missing:  {table.has_missing}")
print(f"units:        {table.units}")


column_names: ['wind_speed', 'temperature', 'solar_power']
n_columns:    3
num_rows:     24
has_missing:  False
units:        ['m/s', 'degC', 'MW']


## Creating directly from a Polars DataFrame

`TimeSeriesTable.from_polars()` wraps an already-normalised Polars DataFrame — useful when data is already in Polars format (e.g. from a database query or Arrow pipeline).

In [4]:
pl_df = pl.DataFrame({
    "valid_time":   pd.date_range("2024-01-15", periods=4, freq="1h", tz="UTC"),
    "wind_speed":   [8.5, 9.2, 10.1, 9.7],
    "temperature":  [-1.5, -1.2, -0.8, -1.0],
}).with_columns(
    pl.col("valid_time").cast(pl.Datetime("us", time_zone="UTC"))
)

table_pl = TimeSeriesTable.from_polars(
    pl_df,
    frequency=Frequency.PT1H,
    units=["m/s", "degC"],
)
table_pl


TimeSeriesTable
┌─────────────────────────────────────────────┐
│  Columns:          wind_speed, temperature  │
│  Shape:            4 × 2                    │
│  Frequency:        PT1H                     │
│  Timezone:         UTC                      │
│  Unit:             m/s, degC                │
├─────────────────────────────────────────────┤
│                    wind_speed  temperature  │
│  2024-01-15 00:00         8.5         -1.5  │
│  2024-01-15 01:00         9.2         -1.2  │
│  2024-01-15 02:00        10.1         -0.8  │
│  2024-01-15 03:00         9.7         -1.0  │
└─────────────────────────────────────────────┘

## Selecting a single column → TimeSeries

`select_column()` extracts one value column as a `TimeSeries`, carrying over its per-column metadata (unit, data type, location). Accepts a column name or zero-based integer index.

In [5]:
ts_wind = table.select_column("wind_speed")
ts_wind

Name,wind_speed
Shape,SIMPLE
Rows,24
Frequency,PT1H
Timezone,Europe/Stockholm
Unit,m/s
Data type,OBSERVATION
valid_time,wind_speed
2024-01-15 00:00,8.15
2024-01-15 01:00,8.56
2024-01-15 02:00,10.45


In [6]:
ts_temp = table.select_column(1)  # zero-based index among value columns
ts_temp

Name,temperature
Shape,SIMPLE
Rows,24
Frequency,PT1H
Timezone,Europe/Stockholm
Unit,degC
Data type,OBSERVATION
valid_time,temperature
2024-01-15 00:00,-2.09
2024-01-15 01:00,-1.66
2024-01-15 02:00,-1.08


## Merging TimeSeries objects into a table

`TimeSeriesTable.from_timeseries()` builds a table from a list of SIMPLE-shape `TimeSeries` objects. Column names come from each series' `name` attribute, and per-column metadata (unit, data_type, location) is derived automatically from each series.

In [7]:
ts_a = TimeSeries.from_pandas(
    df[["valid_time", "wind_speed"]].rename(columns={"wind_speed": "value"}),
    name="wind_speed", unit="m/s", frequency=Frequency.PT1H,
    data_type=DataType.OBSERVATION,
)
ts_b = TimeSeries.from_pandas(
    df[["valid_time", "temperature"]].rename(columns={"temperature": "value"}),
    name="temperature", unit="degC", frequency=Frequency.PT1H,
    data_type=DataType.OBSERVATION,
)
ts_c = TimeSeries.from_pandas(
    df[["valid_time", "solar_power"]].rename(columns={"solar_power": "value"}),
    name="solar_power", unit="MW", frequency=Frequency.PT1H,
    data_type=DataType.OBSERVATION,
)

merged = TimeSeriesTable.from_timeseries(
    [ts_a, ts_b, ts_c],
    frequency=Frequency.PT1H,
)
merged

TimeSeriesTable
┌───────────────────────────────────────────────────────────┐
│  Columns:          wind_speed, temperature, solar_power   │
│  Shape:            24 × 3                                 │
│  Frequency:        PT1H                                   │
│  Timezone:         UTC                                    │
│  Unit:             m/s, degC, MW                          │
│  Data type:        OBSERVATION, OBSERVATION, OBSERVATION  │
├───────────────────────────────────────────────────────────┤
│                    wind_speed  temperature  solar_power   │
│  2024-01-15 00:00        8.15        -2.09          0.0   │
│  2024-01-15 01:00        8.56        -1.66          0.0   │
│  2024-01-15 02:00       10.45        -1.08         2.61   │
│  ...                      ...          ...          ...   │
│  2024-01-15 21:00        5.58        -1.15         1.65   │
│  2024-01-15 22:00        7.53        -1.42          0.0   │
│  2024-01-15 23:00        7.92        -1.96          0.0   │
└───────────────────────────────────────────────────────────┘

Series are joined on `valid_time` using a full outer join — missing observations appear as `null`. Metadata (unit, data_type, location) is derived from each `TimeSeries` unless explicitly overridden in `from_timeseries()`.

## Data access — head, tail, has_missing

In [8]:
print(f"len:         {len(table)}")
print(f"has_missing: {table.has_missing}")
print()
print("head(3):")
print(table.head(3).df)

len:         24
has_missing: False

head(3):
shape: (3, 4)
┌─────────────────────────┬────────────┬─────────────┬─────────────┐
│ valid_time              ┆ wind_speed ┆ temperature ┆ solar_power │
│ ---                     ┆ ---        ┆ ---         ┆ ---         │
│ datetime[μs, UTC]       ┆ f64        ┆ f64         ┆ f64         │
╞═════════════════════════╪════════════╪═════════════╪═════════════╡
│ 2024-01-15 00:00:00 UTC ┆ 8.15       ┆ -2.09       ┆ 0.0         │
│ 2024-01-15 01:00:00 UTC ┆ 8.56       ┆ -1.66       ┆ 0.0         │
│ 2024-01-15 02:00:00 UTC ┆ 10.45      ┆ -1.08       ┆ 2.61        │
└─────────────────────────┴────────────┴─────────────┴─────────────┘


## Coverage bar

`coverage_bar()` renders one bar row per value column — green bins where values are present, gray where null.

In [9]:
table.coverage_bar()

wind_speed   ████████████████████████
temperature  ████████████████████████
solar_power  ████████████████████████
             2024-01-15 00:00  2024-01-15 23:00

## Converting to pandas

`to_pandas()` returns a `pandas.DataFrame` with `valid_time` as the index.

In [10]:
df_out = table.to_pandas()
df_out.head()

,wind_speed,temperature,solar_power
valid_time,,,
2024-01-15 00:00:00+00:00,8.15,-2.09,0.00
2024-01-15 01:00:00+00:00,8.56,-1.66,0.00
2024-01-15 02:00:00+00:00,10.45,-1.08,2.61
2024-01-15 03:00:00+00:00,11.39,-0.73,16.22
2024-01-15 04:00:00+00:00,10.58,-0.36,24.57


## Other conversion methods

The same `to_polars()`, `to_list()`, `to_numpy()`, and `to_pyarrow()` methods are available on `TimeSeriesTable`.
The matching `from_list()`, `from_numpy()`, and `from_pyarrow()` constructors complete the round-trip.
Each method includes/requires `valid_time` and all value columns.

| Method | Direction | Extra dependency |
|---|---|---|
| `to_polars()` | TimeSeriesTable → `pl.DataFrame` | none |
| `to_list()` | TimeSeriesTable → `dict[str, list]` | none |
| `to_numpy()` | TimeSeriesTable → `dict[str, np.ndarray]` | `numpy` |
| `to_pyarrow()` | TimeSeriesTable → `pa.Table` | `pyarrow` |
| `from_list(data, frequency=...)` | `dict[str, list]` → TimeSeriesTable | none |
| `from_numpy(data, frequency=...)` | `dict[str, np.ndarray]` → TimeSeriesTable | `numpy` |
| `from_pyarrow(table, frequency=...)` | `pa.Table` → TimeSeriesTable | `pyarrow` |

In [11]:
print("to_polars():", type(table.to_polars()))
print()
cols = table.to_list()
print("to_list() keys:", list(cols.keys()))
print("to_list() first wind value:", cols["wind_speed"][0])
print()

import numpy as np
arr = table.to_numpy()
print("to_numpy() keys:", list(arr.keys()))
print("to_numpy() wind_speed dtype:", arr["wind_speed"].dtype)
print()

import pyarrow as pa
tbl = table.to_pyarrow()
print("to_pyarrow() schema:", tbl.schema)

to_polars(): <class 'polars.dataframe.frame.DataFrame'>

to_list() keys: ['valid_time', 'wind_speed', 'temperature', 'solar_power']
to_list() first wind value: 8.15

to_numpy() keys: ['valid_time', 'wind_speed', 'temperature', 'solar_power']
to_numpy() wind_speed dtype: float64

to_pyarrow() schema: valid_time: timestamp[us, tz=UTC]
wind_speed: double
temperature: double
solar_power: double


## Constructing from other formats

`from_list()`, `from_numpy()`, and `from_pyarrow()` are the exact inverses — each accepts a `frequency` argument (required) plus optional per-column metadata.

In [12]:
# Roundtrip via list
table_from_list = TimeSeriesTable.from_list(table.to_list(), frequency=table.frequency)
print("from_list() shape:", table_from_list.num_rows, "×", table_from_list.n_columns)

# Roundtrip via numpy
table_from_numpy = TimeSeriesTable.from_numpy(table.to_numpy(), frequency=table.frequency)
print("from_numpy() shape:", table_from_numpy.num_rows, "×", table_from_numpy.n_columns)

# Roundtrip via pyarrow
table_from_arrow = TimeSeriesTable.from_pyarrow(table.to_pyarrow(), frequency=table.frequency)
print("from_pyarrow() shape:", table_from_arrow.num_rows, "×", table_from_arrow.n_columns)

assert table_from_list.df.equals(table.df)
assert table_from_numpy.df.equals(table.df)
assert table_from_arrow.df.equals(table.df)
print("All roundtrips verified.")

from_list() shape: 24 × 3
from_numpy() shape: 24 × 3
from_pyarrow() shape: 24 × 3
All roundtrips verified.


## Spatial filtering

When `locations` are provided (one per column), `TimeSeriesTable` supports geographic queries:

- `filter_columns_by_location(center, radius_km)` — keep columns within a radius
- `filter_columns_by_area(area)` — keep columns inside a `GeoArea` polygon
- `nearest_columns(target, n)` — keep the *n* nearest columns by Haversine distance

In [13]:
# Build a table for three Nordic cities
cities = {
    "stockholm":  GeoLocation(59.33, 18.07),
    "gothenburg": GeoLocation(57.71, 11.97),
    "malmo":      GeoLocation(55.60, 13.00),
}

df_cities = pd.DataFrame({
    "valid_time":  pd.date_range("2024-01-15", periods=24, freq="1h", tz="UTC"),
    "stockholm":   np.random.default_rng(1).normal(5.0, 2.0, 24).round(1),
    "gothenburg":  np.random.default_rng(2).normal(4.5, 2.0, 24).round(1),
    "malmo":       np.random.default_rng(3).normal(4.0, 2.0, 24).round(1),
})

table_geo = TimeSeriesTable.from_pandas(
    df_cities,
    frequency=Frequency.PT1H,
    units=["degC", "degC", "degC"],
    locations=list(cities.values()),
)
display(table_geo)
for name, loc in zip(table_geo.column_names, table_geo.locations):
    print(f"  {name}: {loc.latitude:.2f}°N, {loc.longitude:.2f}°E")

TimeSeriesTable
┌────────────────────────────────────────────────────────────────────────┐
│  Columns:          stockholm, gothenburg, malmo                        │
│  Shape:            24 × 3                                              │
│  Frequency:        PT1H                                                │
│  Timezone:         UTC                                                 │
│  Unit:             degC, degC, degC                                    │
│  Location:         59.33°N, 18.07°E, 57.71°N, 11.97°E, 55.6°N, 13.0°E  │
├────────────────────────────────────────────────────────────────────────┤
│                    stockholm  gothenburg  malmo                        │
│  2024-01-15 00:00        5.7         4.9    8.1                        │
│  2024-01-15 01:00        6.6         3.5   -1.1                        │
│  2024-01-15 02:00        5.7         3.7    4.8                        │
│  ...                     ...         ...    ...                        │
│  2024-01-15 21:00        4.4         4.9    7.1                        │
│  2024-01-15 22:00        7.6         5.2    5.1                        │
│  2024-01-15 23:00        7.0         5.3    3.0                        │
└────────────────────────────────────────────────────────────────────────┘

  stockholm: 59.33°N, 18.07°E
  gothenburg: 57.71°N, 11.97°E
  malmo: 55.60°N, 13.00°E


In [14]:
# Keep columns within 300 km of Copenhagen (55.68°N, 12.57°E)
copenhagen = GeoLocation(55.68, 12.57)
nearby = table_geo.filter_columns_by_location(copenhagen, radius_km=300)
print(f"All columns:              {table_geo.column_names}")
print(f"Within 300 km of Copenhagen: {nearby.column_names}")

All columns:              ['stockholm', 'gothenburg', 'malmo']
Within 300 km of Copenhagen: ['gothenburg', 'malmo']


In [15]:
# Filter columns whose location falls inside a bounding polygon
area = GeoArea.bounding_box(copenhagen, 300.0, name="Copenhagen area")
in_area = table_geo.filter_columns_by_area(area)
print(f"Columns inside 300 km bounding box of Copenhagen: {in_area.column_names}")

Columns inside 300 km bounding box of Copenhagen: ['gothenburg', 'malmo']


In [16]:
# Find the 2 nearest columns to Oslo (59.91°N, 10.75°E)
oslo = GeoLocation(59.91, 10.75)
nearest = table_geo.nearest_columns(oslo, n=2)
print(f"2 nearest columns to Oslo: {nearest.column_names}")

2 nearest columns to Oslo: ['gothenburg', 'stockholm']


## Metadata

`metadata_dict()` returns all metadata fields as a plain dict — useful for serialisation or logging.

In [17]:
import json
print(json.dumps(table.metadata_dict(), indent=2, default=str))

{
  "frequency": "PT1H",
  "timezone": "Europe/Stockholm",
  "num_rows": 24,
  "columns": {
    "wind_speed": {
      "unit": "m/s",
      "description": null,
      "data_type": "OBSERVATION",
      "location": null,
      "timeseries_type": "FLAT",
      "labels": {}
    },
    "temperature": {
      "unit": "degC",
      "description": null,
      "data_type": "OBSERVATION",
      "location": null,
      "timeseries_type": "FLAT",
      "labels": {}
    },
    "solar_power": {
      "unit": "MW",
      "description": null,
      "data_type": "OBSERVATION",
      "location": null,
      "timeseries_type": "FLAT",
      "labels": {}
    }
  }
}


## Summary

- **`from_pandas(df, frequency, units, ...)`** — create a table from a pandas DataFrame with per-column metadata
- **`from_polars(pl_df, frequency, units, ...)`** — wrap a Polars DataFrame directly
- **`from_timeseries([ts1, ts2, ...])`** — merge SIMPLE-shape `TimeSeries` objects; column names come from `ts.name`
- **`select_column(name_or_index)`** — extract one column as a `TimeSeries` with its metadata
- **`head(n)` / `tail(n)` / `has_missing`** — data access and quality checks
- **`coverage_bar()`** — visualise null coverage per column as an SVG (Jupyter) or Unicode blocks (terminal)
- **`to_pandas()`** — export with `valid_time` as the index
- **`to_polars()`, `to_list()`, `to_numpy()`, `to_pyarrow()`** — export to other formats (numpy and pyarrow require the respective packages)
- **Spatial filtering** — `filter_columns_by_location()`, `filter_columns_by_area()`, `nearest_columns()` (requires `locations`)
- **`metadata_dict()`** — all metadata as a serialisable dict
